# E-commerce Medallion Pipeline — Full Run

Runs **Bronze → Silver → Gold** from a [Databricks Repo](https://docs.databricks.com/repos/index.html) clone of:

`https://github.com/RAHUL9868/databricks-medallion-pipeline`

**Prerequisites**
- Repo attached to this cluster (Repos → Pull latest `main`)
- Cluster with Delta Lake (all-purpose or serverless)
- **`source_base_path` must be DBFS** (e.g. `dbfs:/FileStore/ecommerce/data`) — Spark cannot read `file:/tmp/...` on serverless
- Set `generate_sample_data` to `true` to create and upload seed=42 CSVs to DBFS automatically

**After this notebook:** build the SQL Dashboard using `src/dashboard/DASHBOARD_GUIDE.md`.

## 1. Configuration

Adjust widgets, then run the next cells in order.

In [ ]:
dbutils.widgets.text("schema_name", "ecommerce", "Hive schema / database")
dbutils.widgets.text("source_base_path", "dbfs:/FileStore/ecommerce/data", "CSV directory on DBFS (required)")
dbutils.widgets.dropdown("generate_sample_data", "true", ["true", "false"], "Generate seed=42 CSVs and upload to DBFS")
dbutils.widgets.text("catalog", "", "Unity Catalog (optional, leave empty on CE)")
dbutils.widgets.text("run_id", "", "Pipeline run id (optional, UTC timestamp if empty)")

SCHEMA_NAME = dbutils.widgets.get("schema_name").strip()
SOURCE_BASE_PATH = dbutils.widgets.get("source_base_path").strip()
GENERATE_SAMPLE_DATA = dbutils.widgets.get("generate_sample_data") == "true"
CATALOG = dbutils.widgets.get("catalog").strip() or None
RUN_ID = dbutils.widgets.get("run_id").strip() or None

# Spark on Databricks cannot ingest from driver-local file:/tmp paths.
DEFAULT_DBFS_DATA_PATH = "dbfs:/FileStore/ecommerce/data"
if SOURCE_BASE_PATH.startswith("file:") or SOURCE_BASE_PATH.startswith("/tmp"):
    print(
        f"WARNING: '{SOURCE_BASE_PATH}' is not readable by Spark on Databricks. "
        f"Using {DEFAULT_DBFS_DATA_PATH} instead."
    )
    SOURCE_BASE_PATH = DEFAULT_DBFS_DATA_PATH
elif not SOURCE_BASE_PATH.startswith("dbfs:"):
    print(
        f"WARNING: source_base_path should be a dbfs:/ URI on Databricks. "
        f"Using {DEFAULT_DBFS_DATA_PATH} instead."
    )
    SOURCE_BASE_PATH = DEFAULT_DBFS_DATA_PATH

print(f"schema={SCHEMA_NAME}")
print(f"source_base_path={SOURCE_BASE_PATH}")
print(f"generate_sample_data={GENERATE_SAMPLE_DATA}")
print(f"catalog={CATALOG or '(default)'}")

## 2. Attach repository `src` to Python path

Resolves the repo root from this notebook's path under `/Workspace/Repos/...`.

In [ ]:
import os
import sys

notebook_path = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook()
    .getContext()
    .notebookPath()
    .get()
)

# e.g. /Repos/user@domain/databricks-medallion-pipeline/notebooks/run_full_pipeline
repo_rel = os.path.dirname(os.path.dirname(notebook_path))
REPO_ROOT = f"/Workspace{repo_rel}" if not repo_rel.startswith("/Workspace") else repo_rel
SRC_ROOT = os.path.join(REPO_ROOT, "src")

if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

assert os.path.isdir(SRC_ROOT), f"src not found at {SRC_ROOT} — open this notebook from the Git repo"
print(f"REPO_ROOT={REPO_ROOT}")
print(f"SRC_ROOT={SRC_ROOT}")

## 3. (Optional) Generate sample CSVs and copy to DBFS

Skip this section if CSVs are already at `source_base_path`.

In [ ]:
from pathlib import Path

if GENERATE_SAMPLE_DATA:
    from data_generation.generate_sample_data import write_sample_datasets, DEFAULT_SEED

    local_dir = Path("/tmp/ecommerce_medallion_sample_data")
    local_dir.mkdir(parents=True, exist_ok=True)

    print(f"Generating seed={DEFAULT_SEED} CSVs to {local_dir} ...")
    write_sample_datasets(local_dir, seed=DEFAULT_SEED)

    dbutils.fs.mkdirs(SOURCE_BASE_PATH)
    for name in ("customers.csv", "products.csv", "orders.csv"):
        src = f"file:{local_dir / name}"
        dest = f"{SOURCE_BASE_PATH.rstrip('/')}/{name}"
        dbutils.fs.cp(src, dest, True)
        print(f"Uploaded {name} -> {dest}")

    print(f"Ingest will read from DBFS: {SOURCE_BASE_PATH}")
else:
    print(f"Skipping sample data generation; expecting CSVs at {SOURCE_BASE_PATH}")

## 4. Run end-to-end pipeline

Stages: config validation → Bronze ingest → Silver DQ → Gold build → reconciliation → final checks.

In [ ]:
import logging

from config.pipeline_config import load_config
from config.databricks_runtime import prepare_config_source_for_spark
from run_pipeline import configure_logging, run_pipeline

configure_logging("INFO")

config = load_config(
    source_base_path=SOURCE_BASE_PATH,
    catalog=CATALOG,
    schema_name=SCHEMA_NAME,
    run_id=RUN_ID,
)

# Ensure DBFS path + upload from /tmp when on Databricks (serverless-safe)
config = prepare_config_source_for_spark(
    spark,
    config,
    local_csv_dir="/tmp/ecommerce_medallion_sample_data" if GENERATE_SAMPLE_DATA else None,
)

print(f"Pipeline will ingest from: {config.source_base_path}")

summary = run_pipeline(
    config,
    spark=spark,
    generate_sample_data_flag=False,
    validate_sample_data=True,
    strict_sample_row_counts=False,
)

print("\n=== Pipeline summary ===")
print(f"run_id: {summary.run_id}")
print(f"batch_id: {summary.batch_id}")
print(f"elapsed_seconds: {summary.elapsed_seconds:.1f}")
print(f"steps: {summary.steps_completed}")
print(f"bronze_row_counts: {summary.bronze_row_counts}")
print(f"silver_row_counts: {summary.silver_row_counts}")
print(f"gold_row_counts: {summary.gold_row_counts}")

## 5. Quick validation (SQL)

Expected row counts for seed **42**: Bronze orders **100,000**; Gold products **500**; segmentation **4** segments.

In [ ]:
spark.sql(f"USE {SCHEMA_NAME}")

display(spark.sql("""
    SELECT 'bronze_customers' AS table_name, COUNT(*) AS row_count FROM bronze_customers
    UNION ALL SELECT 'bronze_products', COUNT(*) FROM bronze_products
    UNION ALL SELECT 'bronze_orders', COUNT(*) FROM bronze_orders
    UNION ALL SELECT 'silver_orders', COUNT(*) FROM silver_orders
    UNION ALL SELECT 'gold_sales_by_product', COUNT(*) FROM gold_sales_by_product
    UNION ALL SELECT 'gold_customer_segmentation', COUNT(*) FROM gold_customer_segmentation
"""))

display(spark.sql("""
    SELECT check_name, failed_records, pass_percentage
    FROM silver_dq_report
    WHERE failed_records > 0
    ORDER BY failed_records DESC
"""))

## 6. Next steps

1. Open `src/dashboard/dashboard_queries.sql` in the repo.
2. Follow **`src/dashboard/DASHBOARD_GUIDE.md`** to create the Databricks SQL Dashboard.
3. After `git push` from your laptop, **Pull** the repo in Databricks to sync changes.